# 用講的跟 Workflow 對話

語音牽動兩個模組，這一份兩個都示範：

- **Action** — `VoiceAnswerAction` 一次產生兩個頻道：說出口的，和顯示在畫面上的。
- **Perceive** — `VoiceTextPerceive` 把說出來的話變成這一輪的輸入，並在有人開口時中止正在進行的回答。

四段依序回答四個問題：打字問可以用聲音答嗎、用講的怎麼問、聲音和文字怎麼同時出去、講到一半被打斷會怎樣。

**全部走真的端點。** 沒有假物件，看到的每一句話都是模型當場產生、當場合成的。


## 在 Colab 準備環境


In [ ]:
!pip install -q "git+https://github.com/R300-AI/Agentic-SDK.git"

## 填入你的端點

這一份用的端點不是 OpenAI，所以下一格要自己接（見下一節）。如果你用的是 OpenAI，那一格可以整個跳過，直接用 `RealtimeTranscription(api_key=..., model=...)`。


In [ ]:
AZURE_ENDPOINT      = "https://<資源>.cognitiveservices.azure.com"
AZURE_API_KEY       = "<KEY>"
CHAT_BASE_URL       = "https://<資源>/openai/v1/"
CHAT_API_KEY        = "<KEY>"
CHAT_MODEL          = "<部署名稱>"
TRANSCRIBE_MODEL    = "<部署名稱>"
TTS_MODEL           = "<部署名稱>"
REALTIME_API_VERSION = "2025-04-01-preview"
SPEECH_API_VERSION   = "2025-03-01-preview"

## 你的端點不是 OpenAI，所以自己接

**SDK 附的傳輸只會建 `OpenAI` client，不內建任何其他廠商。** 連線方式不同時，覆蓋一個方法就好——開連線之後的一切（session 設定、送音訊、事件分派、回合判定、靜音閘門）全部繼承。

`turn_detection` 決定服務怎麼判斷你講完了：靠固定的靜音長度，或靠模型判斷這個停頓是換氣還是句末。**那是傳輸的參數，不是模組的**——模組只管送什麼上去。


In [ ]:
from openai import AzureOpenAI
from agentic_sdk.audio.realtime import RealtimeTranscription
from agentic_sdk.audio.speech import SpeechOutput


class MyTranscription(RealtimeTranscription):
    """我自己接的端點。SDK 只認 OpenAI，這一家是我接的。"""

    def _open(self):
        return AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_API_KEY,
                           api_version=REALTIME_API_VERSION).beta.realtime.connect(
            model=self._model, extra_query={"intent": "transcription"})


class MySpeech(SpeechOutput):
    def _open_stream(self, text):
        return AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_API_KEY,
                           api_version=SPEECH_API_VERSION
                           ).audio.speech.with_streaming_response.create(
            model=self._model, voice=self._voice, input=text,
            response_format=self._response_format)

## 一、打字問，用聲音答（只動 Action）

語音輸出不需要語音輸入。這一段的 perceive 是最普通的 `PassThroughPerceive`，使用者用打的，Agent 用講的。

注意兩個頻道的差別：`spoken` 是口語、講判斷與理由；`displayed` 是要用看的——條列、型號、數字。**兩者互相補充，不是把畫面唸一遍。**


In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import PassThroughPerceive, VoiceAnswerAction

class Recorded(MySpeech):
    """把合成出來的音訊留下來，這樣 Colab 可以播給你聽。"""
    def __init__(self, **kw):
        self.audio = bytearray()
        self.said = []
        super().__init__(**kw)
    def speak(self, text):
        self.said.append(text)
        for piece in super().speak(text):
            self.audio.extend(piece)
            yield piece

speaking = Recorded(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
workflow = Workflow(workflow_name="打字問、聲音答",
                    perceive=PassThroughPerceive(), action=action)
result = workflow.run("保固多久？我要能貼在說明頁上的版本。")

print("畫面上顯示 :", result.final_message.replace("\n", " / ")[:80])
print("說出口的   :", speaking.said[0][:60])
print("合成音訊   :", len(speaking.audio), "bytes")

## 二、用講的問（加上 Perceive）

沒有麥克風也能示範：用同一個合成端點把問句唸出來，再把那段音訊餵進去。

兩件事值得注意。**安靜的片段不會離開這台機器**——純靜音和說話計費相同，而且會被服務辨識成沒有人說過的字。還有 **`run()` 不必再被告知使用者說了什麼**：語音什麼時候來取決於人什麼時候想講，所以模組先收著。


In [ ]:
import array, time
from agentic_sdk.modules import VoiceTextPerceive

def as_microphone(text, *, quiet_after=2.5, rate_in=24000, rate_out=16000):
    """用合成語音假裝有人在講話，這樣沒有麥克風也能示範。

    尾巴要補靜音。真的麥克風在人停止說話之後還是持續在收，而服務就是靠
    「聽到靜音」判定一句話結束——講完就把音訊切掉，轉寫永遠不會回來。
    """
    pcm = b"".join(MySpeech(model=TTS_MODEL).speak(text))
    samples = array.array("h"); samples.frombytes(pcm)
    step = rate_in / rate_out
    out = array.array("h", (samples[int(i * step)] for i in range(int(len(samples) / step))))
    out.extend([0] * int(rate_out * quiet_after))
    frame = 1600                       # 十分之一秒
    return [out[i:i + frame].tobytes() for i in range(0, len(out), frame)]

listening = MyTranscription(model=TRANSCRIBE_MODEL, language="zh",
                            turn_detection={"type": "semantic_vad", "eagerness": "low"})
perceive = VoiceTextPerceive(transport=listening, speech_threshold=500, hangover_seconds=1.2)

# 從轉寫回呼旁觀就好。pending_input() 是「取用」——印出來就等於用掉了，
# 待會 run() 會拿不到東西。
heard = []
listening.on_transcript(heard.append)

chunks = as_microphone("保固期是多久？")
for chunk in chunks:
    perceive.hear(chunk)               # 安靜的片段不會離開這台機器
    time.sleep(0.1)                    # 麥克風是即時來的
for _ in range(60):                    # 等服務判定這句話講完
    if perceive.pending_input():
        break
    time.sleep(0.2)

print("麥克風片段數 :", len(chunks))
print("聽到的話     :", heard[0] if heard else "（還沒回來）")

speaking = Recorded(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
talking = Workflow(workflow_name="用講的問、用聲音答", perceive=perceive, action=action)
result = talking.run()                 # 不必再告訴它使用者說了什麼

print("畫面上顯示   :", result.final_message.replace("\n", " / ")[:60])
print("說出口的     :", speaking.said[0][:50])

## 三、聲音還在講，畫面還在跑字

`spoken` 這個欄位一寫完就送去合成，**不等整段回覆結束**。回覆愈長，這個提早開口愈有感——下面用時間量給你看。


In [ ]:
written = []
started = []

class Timed(Recorded):
    """記下「開始說話」是這一輪的第幾秒。"""
    def speak(self, text):
        started.append(time.monotonic() - began)
        return super().speak(text)

speaking = Timed(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
workflow = Workflow(workflow_name="邊說邊顯示",
                    perceive=PassThroughPerceive(), action=action)

began = time.monotonic()
stream = workflow.stream("保固多久？請給我可以貼在說明頁上的完整條列。")
for piece in stream:
    written.append(piece)
finished = time.monotonic() - began

print(f"開始說話 : 第 {started[0]:.1f} 秒")
print(f"整段寫完 : 第 {finished:.1f} 秒（共 {len(''.join(written))} 個字）")
print(f"→ 聲音比整段文字早了 {finished - started[0]:.1f} 秒")

## 四、講到一半被打斷

偵測靠的是**語音活動，不是聽懂了什麼**——開口約 600 毫秒就測得到，轉寫要將近四秒，等字就等於繼續講在別人身上。

被打斷之後，記憶留下的是**對方實際收到的那一段**，不是模型寫完的整段。沒收到的部分不會進入下一輪的上下文，否則 Agent 會引用一句沒人聽過的話。

注意 `aborted` 是 `False`：那個旗標是流程自我中止（跳轉上限、逾時）才用的，會被當成錯誤顯示。**有人故意插話不是錯誤。**


In [ ]:
from agentic_sdk.core.cancellation import CancellationToken

listening = MyTranscription(model=TRANSCRIBE_MODEL, language="zh",
                            turn_detection={"type": "semantic_vad", "eagerness": "low"})
perceive = VoiceTextPerceive(transport=listening, speech_threshold=500, hangover_seconds=1.2)

class Interrupted(Recorded):
    """一邊播放，一邊讓使用者在中途開口。"""
    def speak(self, text):
        for index, piece in enumerate(super().speak(text)):
            if index == 3:                        # 播到一小段就插話
                for chunk in barge_in:
                    perceive.hear(chunk)
            yield piece

barge_in = as_microphone("等一下，我不是問這個。", quiet_after=1.0)

speaking = Interrupted(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
talking = Workflow(workflow_name="會被打斷的對話", perceive=perceive, action=action)

for chunk in as_microphone("保固期是多久？"):
    perceive.hear(chunk)
    time.sleep(0.1)
for _ in range(60):
    if perceive.pending_input():
        break
    time.sleep(0.2)

cut = talking.run(cancel=CancellationToken())
print("被打斷了嗎 :", cut.interrupted)
print("這是錯誤嗎 :", cut.aborted)
print("對方收到的 :", (cut.interrupt_payload.get("delivered") or "")[:40])
print("記憶裡留的 :", talking.memory.turns[-1].content[:40])

## SDK 不擷取麥克風，也不播放聲音

上面那個 `as_microphone` 就是在補「擷取」那一端；真實應用換成音訊裝置即可。

輸出同理：要讓人聽見，包一層在 `speak()` 裡把每一段交給音訊裝置再 `yield`——**先播放再 `yield`**，插話時放棄串流才會同時停掉播放與合成。上面的 `Recorded` 就是這個形狀，只是它把音訊存起來而不是播出去。

可執行的完整範例在 repo 的 `examples/voice/desktop_voice_agent.py`，用一個 WAV 當麥克風。
